# 06 — Knowledge Distillation

Distill a compact Student CNN from a trained Teacher using temperature scaling and a weighted combination of hard and soft losses.
This notebook loads neural features, the Teacher model, builds the Student per config, and runs distillation with configurable alpha and temperature.


In [ ]:
# Imports
import os, json, yaml
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix
sns.set_context('talk'); sns.set_style('whitegrid')


In [ ]:
def detect_root():
    cwd = Path.cwd()
    for p in [cwd, cwd.parent, cwd.parent.parent]:
        if all((p/d).exists() for d in ['config','data','scripts','notebooks']):
            return p
    return cwd

ROOT = detect_root()
CONFIG_PATH = ROOT/'config'/'config.yaml'
SPLITS_DIR  = ROOT/'data'/'splits'
NEURAL_DIR  = ROOT/'data'/'processed'/'neural'
PLOTS_DIR   = ROOT/'results'/'plots'; PLOTS_DIR.mkdir(parents=True, exist_ok=True)
EVALS_DIR   = ROOT/'results'/'evaluations'; EVALS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR  = ROOT/'models'/'trained'/'neural'; MODELS_DIR.mkdir(parents=True, exist_ok=True)

cfg = yaml.safe_load(open(CONFIG_PATH,'r'))
dcfg = cfg.get('config',{})
tcfg = cfg.get('training',{})
ncfg = tcfg.get('neural',{})
dist = tcfg.get('distillation',{})
classes = dcfg.get('classes',[])
num_classes = len(classes)
classes


Load neural features and splits

In [ ]:
def read_manifest_stems(path: Path):
    stems=set()
    if not path.exists(): return stems
    for line in open(path,'r'):
        line=line.strip()
        if not line: continue
        rel,_ = line.split(',',1)
        stems.add(Path(rel).stem)
    return stems

split_idx = {
    'train': read_manifest_stems(SPLITS_DIR/'train.txt'),
    'val':   read_manifest_stems(SPLITS_DIR/'val.txt'),
    'test':  read_manifest_stems(SPLITS_DIR/'test.txt'),
}

def collect_neural_for_split(split_stems, class_names):
    X_list, y_list, paths = [], [], []
    name_to_idx = {c:i for i,c in enumerate(class_names)}
    for c in class_names:
        cdir = NEURAL_DIR/c
        if not cdir.exists(): continue
        for f in cdir.glob('*.npy'):
            base=f.stem.split('_seg')[0]
            if base in split_stems:
                arr = np.load(f)
                # Stored as [C, n_mels, T] → channels-last
                if arr.ndim==3: arr = np.transpose(arr,(1,2,0))
                elif arr.ndim==2: arr = arr[:,:,None]
                else: continue
                X_list.append(arr.astype(np.float32))
                y_list.append(name_to_idx[c])
                paths.append(str(f))
    if not X_list: return np.empty((0,)), np.empty((0,),dtype=int), []
    X = np.stack(X_list, axis=0); y = np.array(y_list, dtype=int)
    return X, y, paths

X_train, y_train, _ = collect_neural_for_split(split_idx['train'], classes)
X_val,   y_val,   _ = collect_neural_for_split(split_idx['val'], classes)
X_test,  y_test,  _ = collect_neural_for_split(split_idx['test'], classes)
X_train.shape, X_val.shape, X_test.shape


Student model and Distiller

In [ ]:
def build_student_cnn(input_shape, num_classes, cfg):
    filters = cfg.get('filters',[16,32])
    kernel  = cfg.get('kernel_size',[3,3])
    pool    = cfg.get('pool_size',[2,2])
    dropout = float(cfg.get('dropout',0.3))
    dense   = cfg.get('dense_units',[64])
    x_in = keras.layers.Input(shape=input_shape); x = x_in
    for i,f in enumerate(filters):
        k = kernel[i] if i<len(kernel) else 3; p = pool[i] if i<len(pool) else 2
        x = keras.layers.Conv2D(f,(k,k),padding='same')(x)
        x = keras.layers.BatchNormalization()(x)
        x = keras.layers.ReLU()(x)
        x = keras.layers.MaxPool2D((p,p))(x)
        x = keras.layers.Dropout(dropout)(x)
    x = keras.layers.Flatten()(x)
    for u in dense:
        x = keras.layers.Dense(u, activation='relu')(x)
        x = keras.layers.Dropout(dropout)(x)
    out = keras.layers.Dense(num_classes, activation='softmax')(x)
    return keras.Model(x_in, out, name='student_cnn')

class Distiller(keras.Model):
    def __init__(self, student, teacher, alpha=0.1, temperature=3.0):
        super().__init__()
        self.student = student; self.teacher = teacher
        self.alpha=float(alpha); self.temperature=float(temperature)
        self.ce = keras.losses.CategoricalCrossentropy()
        self.kld= keras.losses.KLDivergence()
        self.acc= keras.metrics.CategoricalAccuracy(name='accuracy')
    def compile(self, optimizer):
        super().compile(optimizer=optimizer)
    def train_step(self, data):
        x,y = data
        t_logits = self.teacher(x, training=False)
        with tf.GradientTape() as tape:
            s_logits = self.student(x, training=True)
            s_loss = self.ce(y, s_logits)
            T = self.temperature
            t_soft = tf.nn.softmax(t_logits/T, axis=-1)
            s_soft = tf.nn.softmax(s_logits/T, axis=-1)
            kd_loss = self.kld(t_soft, s_soft) * (T*T)
            loss = self.alpha*s_loss + (1.0-self.alpha)*kd_loss
        grads = tape.gradient(loss, self.student.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.student.trainable_variables))
        self.acc.update_state(y, s_logits)
        return {'loss':loss, 's_loss':s_loss, 'kd_loss':kd_loss, 'accuracy':self.acc.result()}
    def test_step(self, data):
        x,y = data
        s_logits = self.student(x, training=False)
        s_loss = self.ce(y, s_logits)
        self.acc.update_state(y, s_logits)
        return {'loss':s_loss, 'accuracy':self.acc.result()}


Training and evaluation

In [ ]:
# Input shape is channels-last (n_mels, T, C)
input_shape = X_train.shape[1:]
y_train_oh = keras.utils.to_categorical(y_train, num_classes)
y_val_oh   = keras.utils.to_categorical(y_val, num_classes)
y_test_oh  = keras.utils.to_categorical(y_test, num_classes)

# Hyperparameters
batch_size = int(tcfg.get('batch_size',32))
epochs = int(tcfg.get('epochs',100))
alpha = float(dist.get('alpha',0.1))
temperature = float(dist.get('temperature',3.0))
lr = float(tcfg.get('learning_rate',1e-3))

# Load teacher and build student
teacher_path = MODELS_DIR/'teacher_cnn.h5'
if not teacher_path.exists():
    raise FileNotFoundError(f'Teacher model not found at {teacher_path}. Train 04 first.')
teacher = keras.models.load_model(teacher_path)
teacher.trainable = False
student = build_student_cnn(input_shape, num_classes, ncfg.get('student_cnn',{}))

# Distiller compile
distiller = Distiller(student, teacher, alpha=alpha, temperature=temperature)
distiller.compile(optimizer=keras.optimizers.Adam(lr))

# Callbacks
early_pat = int(tcfg.get('early_stopping_patience',15))
rlrop_pat = int(tcfg.get('reduce_lr_patience',10))
rlrop_fac = float(tcfg.get('reduce_lr_factor',0.5))
callbacks=[
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=early_pat, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=rlrop_fac, patience=rlrop_pat, verbose=1),
]

hist = distiller.fit(
    X_train, y_train_oh,
    validation_data=(X_val, y_val_oh),
    epochs=epochs,
    batch_size=batch_size,
    callbacks=callbacks,
    verbose=1,
)

# Save distilled student
save_path = MODELS_DIR/'student_cnn_distilled_nb.h5'
student.save(save_path)
print('Saved distilled student to', save_path)

# Evaluate on test
y_pred = np.argmax(student.predict(X_test, batch_size=batch_size), axis=1)
print(classification_report(y_test, y_pred, target_names=classes, output_dict=False))
cm = confusion_matrix(y_test, y_pred, labels=list(range(num_classes)))
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title('Student (Distilled) — Test Confusion')
plt.tight_layout(); plt.show()
